In [3]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader, random_split
import torch
from torch import nn
import torch.nn.functional as F

/mnt/9a2c028e-6c0b-4d1a-ada1-2349e5ef78f7/projects/pytorch-transformer/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ds_raw = load_dataset('opus_books', "en-es", split='train')

In [5]:
type(ds_raw)

datasets.arrow_dataset.Dataset

In [6]:
a = [ds_raw[i]['translation']['es'] for i in range(100)]

In [7]:
import re

def clean_text_es(text):
    text = text.replace('\n', ' ').replace('\r', ' ')
    spanish_pattern = r'[^a-zA-ZáéíóúÁÉÍÓÚñÑüÜçÇ\s]'
    text = re.sub(spanish_pattern, '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_text_en(text):
    text = re.sub(r'[\n\r]+', ' ', text)  # Remove newlines
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Keep only letters and spaces
    text = re.sub(r'\s+', ' ', text).strip()  # Clean whitespace
    return text

def transform(item):
    en_text = clean_text_en(item['translation']['en'])
    es_text = clean_text_es(item['translation']['es'])
    item['translation']['en'] = en_text
    item['translation']['es'] = es_text

    return item
    
new_ds = ds_raw.map(transform).filter(lambda item: len(item['translation']['en']) <= 100)

In [8]:
# print('\n'.join([new_ds[i]['translation']['en'] + '\t' + new_ds[i]['translation']['es'] for i in range(100)]))

In [9]:
count = 0
i = 2
en_str = ""
es_str = ""
for i in range(len(new_ds)):
    en_str += new_ds[i]['translation']['en']
    es_str += new_ds[i]['translation']['es']

In [10]:
en_chrs = list(set(en_str))
en_chrs = ['<sos>', '<eos>', '<pad>'] + en_chrs
es_chrs = list(set(es_str))
es_chrs = ['<sos>', '<eos>', '<pad>'] + es_chrs

print(len(en_str))
print(en_chrs)
print(len(es_str))
print(es_chrs)

en_chrs_size = len(en_chrs)
es_chrs_size = len(es_chrs)
en_itoc = {i: en_chrs[i] for i in range(len(en_chrs))}
en_ctoi = {en_chrs[i]: i for i in range(len(en_chrs))}
es_itoc = {i: es_chrs[i] for i in range(len(es_chrs))}
es_ctoi = {es_chrs[i]: i for i in range(len(es_chrs))}

for ch in en_chrs:
    assert en_itoc[en_ctoi[ch]] == ch
for ch in es_chrs:
    assert es_itoc[es_ctoi[ch]] == ch
en_encode = lambda s: [en_ctoi[c] for c in s]
en_decode = lambda l: ''.join([en_itoc[i] for i in l])
es_encode = lambda s: [es_ctoi[c] for c in s]
es_decode = lambda l: ''.join([es_itoc[i] for i in l])
assert en_decode(en_encode(new_ds[10]['translation']['en'])) == new_ds[10]['translation']['en']
assert es_decode(es_encode(new_ds[10]['translation']['es'])) == new_ds[10]['translation']['es']

print(f"Vocab sizes: {en_chrs_size}, {es_chrs_size}")

2626501
['<sos>', '<eos>', '<pad>', 'G', 'l', 'b', 'X', 'Z', 'L', 'x', 'k', 'T', 'n', 'S', 'z', 'R', 'q', 'O', 'W', 'I', 'm', 'a', 'w', 'Y', 'e', ' ', 'u', 't', 'D', 'B', 'P', 'N', 'Q', 'y', 'M', 'h', 'g', 'E', 'C', 'K', 'v', 's', 'H', 'A', 'c', 'j', 'J', 'F', 'V', 'r', 'o', 'f', 'i', 'd', 'U', 'p']
2820109
['<sos>', '<eos>', '<pad>', 'G', 'l', 'b', 'X', 'Z', 'Ó', 'L', 'x', 'Ç', 'k', 'T', 'n', 'S', 'z', 'R', 'O', 'q', 'W', 'I', 'á', 'm', 'É', 'Y', 'a', 'w', 'e', ' ', 'u', 't', 'N', 'D', 'B', 'P', 'ñ', 'Q', 'y', 'ó', 'ú', 'Í', 'Á', 'M', 'h', 'ü', 'E', 'g', 'C', 'é', 'K', 's', 'v', 'H', 'A', 'ç', 'c', 'J', 'j', 'F', 'V', 'Ú', 'r', 'o', 'f', 'í', 'i', 'd', 'U', 'p']
Vocab sizes: 56, 70


In [11]:
train_ds_size = int(0.9 * len(new_ds))
val_ds_size = len(new_ds) - train_ds_size
train_ds_raw, val_ds_raw = random_split(new_ds, [train_ds_size, val_ds_size])

In [12]:
class EnEsDataset(Dataset):

    def __init__(self, ds):
        super().__init__()
        self.ds = ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        src_target_pair = self.ds[idx]
        src_text = src_target_pair['translation']['en']
        tgt_text = src_target_pair['translation']['es']

        src_tokens = torch.tensor([en_ctoi['<sos>']] + en_encode(src_text) + [en_ctoi['<eos>']])
        tgt_tokens = torch.tensor([es_ctoi['<sos>']] + es_encode(tgt_text) + [es_ctoi['<eos>']])

        return {
            "src_tokens": src_tokens,
            "tgt_tokens": tgt_tokens,
            "src_text": src_text,
            "tgt_text": tgt_text,
        }

In [13]:
train_ds = EnEsDataset(train_ds_raw)

In [14]:
assert en_ctoi['<pad>'] == es_ctoi['<pad>']
def collate_fn(batch):
    batch_src_tokens = [item['src_tokens'] for item in batch]
    batch_tgt_tokens = [item['tgt_tokens'] for item in batch]
    batch_src_tokens = nn.utils.rnn.pad_sequence(batch_src_tokens, padding_value=es_ctoi['<pad>'], batch_first=True)
    batch_tgt_tokens = nn.utils.rnn.pad_sequence(batch_tgt_tokens, padding_value=es_ctoi['<pad>'], batch_first=True)
    return {
        'src_tokens': batch_src_tokens,
        'tgt_tokens': batch_tgt_tokens
    }

In [15]:
class Encoder(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        # 2 layers
        self.lstm0 = nn.LSTMCell(input_size, hidden_size)
        self.lstm1 = nn.LSTMCell(hidden_size, hidden_size)

    def forward(self, inputs):
        inputs = inputs.transpose(0, 1) # (B, T, input_size) -> (T, B, input_size)
        hc_0, hc_1 = None, None
        for x in inputs:
            hc_0 = self.lstm0(x, hc_0)
            hc_1 = self.lstm1(hc_0[0], hc_1)
        return hc_0, hc_1

class Decoder(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        # 2 layers
        self.lstm0 = nn.LSTMCell(input_size, hidden_size)
        self.lstm1 = nn.LSTMCell(hidden_size, hidden_size)
        

    def forward(self, inputs, encoder_state):
        inputs = inputs.transpose(0, 1) # (B, T, input_size) -> (T, B, input_size)
        hc_0, hc_1 = encoder_state
        outputs = []
        for x in inputs:
            hc_0 = self.lstm0(x, hc_0)
            hc_1 = self.lstm1(hc_0[0], hc_1)
            outputs.append(hc_1[0])
        return torch.stack(outputs).transpose(0, 1), (hc_0, hc_1) # (T, B, hidden_size) -> (B, T, hidden_size)
        
class Seq2Seq(nn.Module):
    def __init__(self, in_vocab_size, in_embd_size, out_vocab_size, out_embd_size, hidden_size):
        super().__init__()
        # project to out vocab
        self.in_embd = nn.Embedding(in_vocab_size, in_embd_size)
        self.encoder = Encoder(in_embd_size, hidden_size)
        self.out_embd = nn.Embedding(out_vocab_size, out_embd_size)
        self.decoder = Decoder(out_embd_size, hidden_size)
        self.proj = nn.Linear(hidden_size, out_vocab_size)

    def clip_gradient(self):
        params = [p for p in self.parameters() if p.requires_grad]
        norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in params))
        if norm > 1:
            for p in params:
                p.grad[:] /= norm

    def forward(self, x, y):
        x = self.in_embd(x) # (B, T) -> (B, T, in_embd_size)
        encoder_out = self.encoder(x)
        y = self.out_embd(y) # (B, T) -> (B, T, out_embd_size)
        out, _ = self.decoder(y, encoder_out)
        out = self.proj(out)
        return out        

    @torch.no_grad()
    def generate(self, en_str, max_token=100):
        x = self.in_embd(torch.tensor([en_encode(en_str)]))
        decoder_state = self.encoder(x) # ((h0,c0),(h1,c1)) -> all are vectors (hidden_size)
        gen_ids = [es_ctoi['<sos>']]
        for _ in range(max_token):
            decoder_in = self.out_embd(torch.tensor([[gen_ids[-1]]])) # (1, 1, out_embd_size)
            out, decoder_state = self.decoder(decoder_in, decoder_state) # out -> (1, 1, hidden_size), decoder_state -> ((h0,c0),(h1,c1))
            out = self.proj(out) # out -> (1, 1, out_vocab_size)
            gen_ids.append(torch.argmax(out[0][0]).item())
            if gen_ids[-1] == es_ctoi['<eos>']:
                break

        return es_decode(gen_ids)


In [77]:
model = Seq2Seq(in_vocab_size=en_chrs_size, in_embd_size=128, out_vocab_size=es_chrs_size, out_embd_size=128, hidden_size=128)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                          collate_fn=collate_fn)
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)


model.apply(init_weights)

Seq2Seq(
  (in_embd): Embedding(56, 128)
  (encoder): Encoder(
    (lstm0): LSTMCell(128, 128)
    (lstm1): LSTMCell(128, 128)
  )
  (out_embd): Embedding(70, 128)
  (decoder): Decoder(
    (lstm0): LSTMCell(128, 128)
    (lstm1): LSTMCell(128, 128)
  )
  (proj): Linear(in_features=128, out_features=70, bias=True)
)

In [78]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"The model has {count_parameters(model):,} trainable parameters")

The model has 553,542 trainable parameters


In [80]:

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
train_iter = iter(train_loader)
for batch in train_iter:
    X, Y = batch['src_tokens'], batch['tgt_tokens']
    Y_hat = model(X, Y[:,:-1])
    loss = F.cross_entropy(Y_hat.reshape(-1, Y_hat.shape[-1]), Y[:, 1:].reshape(-1), ignore_index=pad_id)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    print(loss.item())

2.9914469718933105
3.029297113418579
3.0443150997161865
3.008549928665161
3.0146846771240234
3.0152552127838135
3.0294437408447266
3.0662121772766113
3.016429901123047
3.0325286388397217
3.016974449157715
3.0567078590393066
2.9953300952911377
3.0260136127471924
3.0224010944366455
3.0031728744506836
2.9955358505249023
3.0317189693450928
3.013046979904175
3.025205612182617
3.12699818611145
2.9938769340515137
3.0092480182647705
3.014310836791992
3.0061466693878174
2.9904541969299316
3.019381523132324
3.012657642364502
2.998805046081543
3.016388416290283
2.9900014400482178
2.995537281036377
2.9968276023864746
3.0322136878967285
3.0280990600585938
3.0199368000030518
2.981518268585205
2.9982850551605225
3.061898708343506
3.029985189437866
3.019470691680908
3.0391860008239746
3.00426983833313
2.999861240386963
3.021273374557495
3.032014846801758
2.9965078830718994
3.006317615509033
2.9991567134857178
2.990654468536377
3.0316309928894043
3.0137200355529785
3.03233003616333
3.0298972129821777
3

In [85]:
model.generate('How', 100)

'<sos>o e ue de de le de le de le de le de le de le de le de le de le de le de le de le de le de le de le '

In [100]:
((h0,c0), (h1,c1)) = decoder_state
h0.shape

torch.Size([1, 128])

In [13]:
lstm = nn.LSTMCell(10, 20)
x = torch.randn(5, 10) # (B, input_size)
h, c = lstm(x, None)
h.shape, c.shape

(torch.Size([5, 20]), torch.Size([5, 20]))

In [14]:
rnn = nn.LSTM(10, 20, batch_first=True)
inputs = torch.randn(5, 7, 10)
outputs, (h, c) = rnn(inputs)

In [24]:
outputs[:,-1,:] == h[0]

tensor([[True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True]])

In [15]:
print(outputs.shape)
print(h.shape)
print(c.shape)

torch.Size([5, 7, 20])
torch.Size([1, 5, 20])
torch.Size([1, 5, 20])
